In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent  # parent of 'notebooks' directory
sys.path.append(str(project_root))
print("Project root added to sys.path:", project_root)

Project root added to sys.path: c:\Users\chakr\Downloads\projects\deploy_rag


In [2]:
from pathlib import Path
from langchain.schema import Document
from multi_doc_chat.utils.faiss_manager import FaissManager

c:\Users\chakr\anaconda3\envs\chat_pdf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
index_dir = Path("faiss_index")

In [4]:
manager = FaissManager(index_dir=index_dir)

{"timestamp": "2025-11-07T14:40:33.007717Z", "level": "info", "event": "Running in Prod env"}
{"timestamp": "2025-11-07T14:40:33.008741Z", "level": "info", "event": "Loaded API KEYS"}
{"keys": {"GROQ_API_KEY": "gsk_oc....."}, "timestamp": "2025-11-07T14:40:33.008741Z", "level": "info", "event": "API Keys loaded"}
{"config_keys": ["embedding_model", "llm"], "timestamp": "2025-11-07T14:40:33.010680Z", "level": "info", "event": "Config YAML loaded"}
Use pytorch device_name: cpu
Load pretrained SentenceTransformer: BAAI/bge-small-en


In [5]:
texts = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks for representation learning.",
    "FAISS is a library for efficient similarity search."
]
metadatas = [{"source": "note1.txt"}, {"source": "note2.txt"}, {"source": "note3.txt"}]

In [6]:
# Load or create index
manager.load_or_create(texts=texts, metadatas=metadatas)

Loading faiss with AVX2 support.
Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
Loading faiss.
Successfully loaded faiss.


When a new document is added in FAISS, entry will be made in ingested_meta.json

In [7]:
new_docs = [
    Document(page_content="Transformers have revolutionized NLP.", metadata={"source": "note4.txt"}),
    Document(page_content="Transformers have revolutionized NLP.", metadata={"source": "note5.txt"})
]
added_count = manager.add_documents(new_docs)
print(f"✅ Added {added_count} new documents.")

✅ Added 0 new documents.


In [8]:
query = "What is FAISS?"
results = manager.vs.similarity_search(query, k=2)
for r in results:
    print(f"- {r.page_content} (source: {r.metadata.get('source')})")

- FAISS is a library for efficient similarity search. (source: note3.txt)
- Machine learning is a subset of artificial intelligence. (source: note1.txt)


In [18]:
deleted_count = manager.delete_documents(["note5.txt"])
print(f"🗑️ Deleted {deleted_count} documents.")


🗑️ Deleted 0 documents.


In [10]:
results_after = manager.vs.similarity_search("Deep learning", k=8)
for r in results_after:
    print(f"- {r.page_content} (source: {r.metadata.get('source')})")

- Deep learning uses neural networks for representation learning. (source: note2.txt)
- Machine learning is a subset of artificial intelligence. (source: note1.txt)
- Transformers have revolutionized NLP. (source: note4.txt)
- Transformers have revolutionized NLP. (source: note5.txt)
- FAISS is a library for efficient similarity search. (source: note3.txt)


In [16]:
deleted = manager.delete_by_metadata({"source": "note4.txt"})
print("deleted:", deleted)

deleted: 1


In [17]:
from pathlib import Path
import json

meta_path = Path("faiss_index") / "ingested_meta.json"   # adjust if your index_dir is different
print("meta file:", meta_path.resolve())
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    for k in meta.get("rows", {}):
        print("-", k)
else:
    print("No ingested_meta.json found at", meta_path)


meta file: C:\Users\chakr\Downloads\projects\deploy_rag\notebooks\faiss_index\ingested_meta.json
- note5.txt::
